## Group 7

**Group Number:** 7

| Member | Student ID |
|---|---|
| ZHANG SHIHAN | A0330526M |
| FANG XUYI | A0353826W |
| CHEN JUNXIAO | A0356288M |
| CHOOG SHENG HONG | Unavailable — unable to contact; instructor informed |

### Planned responsibilities

The following describes the agreed task allocation, not a record of completed contributions. Final contribution statements will be revised after each member's actual work has been confirmed.

- **ZHANG SHIHAN:** Responsible for project coordination and code integration; the Streamlit interface and reasoning trace presentation; application deployment; overall testing of all five Sudoku puzzles; FC/BC runtime comparison; Notebook organization and final submission checks; and conceptual questions Q2 and Q5.
- **CHEN JUNXIAO:** Responsible for encoding the general propositional and Horn knowledge bases, full-grid Sudoku solving with forward chaining, and conceptual questions Q1 and Q4.
- **FANG XUYI:** Responsible for the backward-chaining algorithm and full-grid Sudoku solving, algorithm correctness and cyclic-dependency testing, and conceptual question Q3.
- **CHOOG SHENG HONG:** Could not be contacted and did not contribute to the assignment. The instructor has been informed.

**Actual contributions:** Awaiting confirmation for ZHANG SHIHAN, CHEN JUNXIAO, and FANG XUYI. CHOOG SHENG HONG did not contribute, as confirmed by the group representative.

**Submission status:** Final actual contribution statements for the three participating members remain to be confirmed. CHOOG SHENG HONG's student ID could not be obtained; his lack of contact and non-participation have been reported to the instructor. This does not assert that the instructor has approved removing him from the group. The verified public app URL and deployment checks are recorded below.


**Review draft:** This notebook describes the teammate's supplied solver and app.
Their five supplied puzzles passed separate solver checks, but fixes to generic
Horn-query caching, repeated premises, long dependency chains, and app state
clearing are pending. It is not the final submission version.

**Student ID check:** FANG XUYI is listed above using the previously supplied
A0353826W. The teammate's notebook instead lists A0353836W; this discrepancy still
requires confirmation before submission.


# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), ' ') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
[' ', 3, ' ', ' ', ' ', ' ', ' ', ' ', ' ']
[2, ' ', ' ', ' ', ' ', 7, 8, 6, ' ']
[5, 8, ' ', 2, 6, ' ', ' ', 3, ' ']
[7, 5, ' ', ' ', ' ', ' ', ' ', 8, ' ']
[' ', ' ', ' ', ' ', 7, ' ', 5, ' ', 4]
[' ', ' ', ' ', 5, 3, ' ', ' ', 9, 6]
[' ', 1, 2, ' ', ' ', 9, ' ', ' ', ' ']
[6, 4, ' ', ' ', 5, 8, 9, ' ', ' ']
[' ', ' ', ' ', ' ', 2, 3, ' ', ' ', ' ']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [5]:
# Attempt (a): try solving the puzzle using pl_resolution / tt_entails on the
# general KB. Left commented out because it is expected to take an
# impractically long time (pl_resolution) or be outright infeasible
# (tt_entails) on the full grid.
#
# Uncomment a few lines at a time and give each at most ~30 seconds; use
# Kernel > Interrupt if it hasn't returned by then.
#
general_kb = build_general_kb(n, box_h, box_w, givens)
r, c, v = 1, 1, givens.get((1, 1), 1)  # try one cell/value pair
query = atom('Is', r, c, v)

# print(pl_resolution(general_kb, query))    
# print(tt_entails(associate('&', general_kb.clauses), query)) 

### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


The teammate reported that `build_general_kb` produced 10,317 clauses in
approximately 0.3 seconds and that the inference attempt did not return within
minutes. That reported timing is not independently established by the outputs in
this notebook. The teacher asks for bounded attempts of about 30 seconds; stop an
attempt at that point rather than waiting indefinitely.

A previous, separate experiment on 25 September used the given-value query
`Is1_2_3` with unchanged supplied algorithms. Neither returned before an automatic
process timeout (30.025 s for resolution; 30.028 s for model checking, including
startup and KB construction). Those timings belong to that earlier query and
execution method, not to the teacher-example query `Is1_1_1` above.

The two direct calls above remain commented out, as in the teacher's template.
For the final manual experiment, enable one at a time, use Kernel > Interrupt if
it has not returned after about 30 seconds, and record that actual observation.
A timeout or interruption is not a logical False result.


**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [6]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [7]:
# Verify solve_full_grid_fc against the puzzle's known solution. Then check
# pl_bc_entails directly, for completeness (it finds the correct value) and
# soundness (it never wrongly confirms an incorrect one), before timing
# solve_full_grid_bc against the same solution.
#
import time

t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']


definite_kb = build_definite_kb(n, box_h, box_w, givens)

# Completeness: pl_bc_entails must find the correct value for every cell.
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == True

# Soundness: pl_bc_entails must not also confirm any incorrect value.
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) == False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f"solve_full_grid_fc: {fc_time:.2f}s")
for r in range(1,n+1):
    for c in range(1,n+1):
        print(solved.get((r,c)),end=' ')
    print('')
print(f"solve_full_grid_bc: {bc_time:.2f}s")
for r in range(1,n+1):
    for c in range(1,n+1):
        print(solved_bc.get((r,c)),end=' ')
    print('')

solve_full_grid_fc: 13.25s
1 3 6 9 8 5 4 7 2 
2 9 4 3 1 7 8 6 5 
5 8 7 2 6 4 1 3 9 
7 5 1 4 9 6 2 8 3 
3 6 9 8 7 2 5 1 4 
4 2 8 5 3 1 7 9 6 
8 1 2 6 4 9 3 5 7 
6 4 3 7 5 8 9 2 1 
9 7 5 1 2 3 6 4 8 
solve_full_grid_bc: 0.49s
1 3 6 9 8 5 4 7 2 
2 9 4 3 1 7 8 6 5 
5 8 7 2 6 4 1 3 9 
7 5 1 4 9 6 2 8 3 
3 6 9 8 7 2 5 1 4 
4 2 8 5 3 1 7 9 6 
8 1 2 6 4 9 3 5 7 
6 4 3 7 5 8 9 2 1 
9 7 5 1 2 3 6 4 8 


In [8]:
# Validate the remaining provided puzzles with the same imported solver functions.
for index, entry in enumerate(puzzle_pool[1:], 2):
    start = time.perf_counter()
    fc_grid = solve_full_grid_fc(n, box_h, box_w, entry['givens'])
    fc_elapsed = time.perf_counter() - start
    start = time.perf_counter()
    bc_grid = solve_full_grid_bc(n, box_h, box_w, entry['givens'])
    bc_elapsed = time.perf_counter() - start
    assert fc_grid == bc_grid == entry['solution']
    kb = build_definite_kb(n, box_h, box_w, entry['givens'])
    for (r, c), correct in entry['solution'].items():
        for value in range(1, n + 1):
            assert pl_bc_entails(kb, atom('Is', r, c, value)) == (value == correct)
    print(f'Puzzle {index}: FC={fc_elapsed:.3f}s; BC={bc_elapsed:.3f}s; '
          'both full grids and 729 BC candidate queries PASS')
print('All five puzzles: both full grids and 3,645 BC candidate queries PASS')


Puzzle 2: FC=13.056s; BC=0.535s; both full grids and 729 BC candidate queries PASS


Puzzle 3: FC=12.664s; BC=0.446s; both full grids and 729 BC candidate queries PASS


Puzzle 4: FC=11.372s; BC=0.454s; both full grids and 729 BC candidate queries PASS


Puzzle 5: FC=10.359s; BC=0.591s; both full grids and 729 BC candidate queries PASS
All five puzzles: both full grids and 3,645 BC candidate queries PASS


## Part B: Conceptual Questions

The answers below describe the received teammate implementation and distinguish known issues from validated behavior.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**General CNF representation.** Let $I_{rcv}$ mean that cell $(r,c)$ contains $v$.
Each cell receives one clause $I_{rc1}\lor\cdots\lor I_{rcn}$. For every distinct
pair $u<v$ in that cell we add $\neg I_{rcu}\lor\neg I_{rcv}$. Together these
say exactly one value is assigned. For two different cells $a,b$ sharing a row,
column, or box, add $\neg I_{av}\lor\neg I_{bv}$ for each $v$. Finally, each
given becomes a positive unit clause. These clauses directly encode conditions
1–6. Calling `PropKB.tell` converts to CNF; our clauses are already in CNF.
We deduplicate overlapping row/box or column/box constraints by processing each
unordered peer-cell pair once. The 9×9 encoding uses 729 Is atoms, 81 at-least-one
clauses, 2,916 within-cell exclusions, 7,290 peer exclusions, and the given facts.

**Definite representation.** A definite clause has exactly one positive literal
(or equivalently a positive conjunctive body implying one positive head).
General Horn clauses allow at most one, but the supplied `PropDefiniteKB`
requires definite clauses. A multi-positive at-least-one disjunction cannot be
directly inserted. We introduce the positive atom $N_{rcv}$, meaning that $v$
has been explicitly excluded; it is separate from logical negation.

The operational encoding is:

1. Given $(r,c)=v$: assert $I_{rcv}$.
2. Same cell, $u\ne v$: $I_{rcv}\Rightarrow N_{rcu}$.
3. Distinct peer cells $a,b$: $I_{av}\Rightarrow N_{bv}$ and
   $I_{bv}\Rightarrow N_{av}$ for every value $v$.
4. Last candidate: $\bigwedge_{u\ne v}N_{rcu}\Rightarrow I_{rcv}$.

For example, once a cell's eight other numbers are excluded, rule 4 places the
remaining number. Rules 2–3 propagate the at-most-one and row/column/box
constraints; rule 4 is the elimination consequence of the at-least-one constraint.
All rule premises and heads are positive atoms, so `tell` accepts them. For 9×9
there are 5,832 same-cell rules, 14,580 directed peer rules, 729 last-candidate
rules, and the givens.

**Scope of the representation.** These are sound deduction rules relative to the
Sudoku constraints, not a logically equivalent Horn translation of the full CNF.
They cannot express arbitrary choices or guarantee a placement in every cell of
an arbitrary puzzle. Both inference algorithms are complete for this Horn KB,
but this particular rule set is not a complete method for all Sudoku puzzles.
If elimination stalls, the solver reports an unresolved cell rather than guessing.
A failed Is query never causes a Not fact to be asserted.

### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


Neither supplied algorithm is a practical default for the 9×9 board. Soundness
and completeness describe logical correctness, not a usable time or memory bound.
The resource-limited observations above measure one atomic query, already enough
to demonstrate the implementation costs.

- **Model checking (`tt_entails`).** There are 729 Boolean Is variables, so exhaustive
  enumeration has up to $2^{729}\approx2.82\times10^{219}$ leaves. In this supplied
  implementation `tt_check_all` evaluates the KB only after assigning every variable;
  it does not prune partial assignments using Sudoku constraints. Each leaf also
  has a nontrivial formula-evaluation cost. Short-circuiting can sometimes find a
  countermodel early, but a genuinely entailed query still requires exhaustive
  checking. A 30-second cutoff is not a logical False result.
- **Resolution (`pl_resolution`).** The code first materializes every unordered
  pair among $m$ clauses, using $m(m-1)/2=O(m^2)$ list entries before resolving
  the first pair. With 10,317 original clauses and the negated query, the first
  round alone has 53,225,403 pairs. This creates a large memory demand even for a
  query whose value is given. Subsequent rounds can add many resolvents and repeat
  the pair construction. The worst-case number of distinct propositional clauses
  also grows exponentially with the number of variables.

The recorded attempts were stopped after about 30 seconds without a Boolean
answer; see the observation notes for their original execution method. Stopping
an experiment does not establish non-entailment. Following the teacher's template,
the cell now contains direct library calls to run separately and interrupt if
needed; it does not replace or modify either algorithm. Full-grid solving uses
the elimination Horn KB instead.


### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Design.** The supplied student code performs goal-directed backward chaining
with shared tables. It indexes rules by conclusion and stores initial facts.
Alternative rules for a goal are OR choices; all premises of a selected rule
must be proved (AND). A repeated expanded goal is not accepted as true: rules
waiting for it can be resumed when an external fact-grounded proof becomes available.

```text
PROVE(q, depth):
    if q is a fact or already proved: return True
    if q was expanded: return False for this visit
    if depth >= 64: defer q; return False for this visit
    mark q expanded
    for each body implying q:
        missing := body premises not yet known true
        remaining[(q, body)] := number of missing premises
        if missing is empty: MARK_PROVEN(q); return True
        register the rule in each missing premise's waiting set
        for p in missing: PROVE(p, depth + 1)
        if q is now proved: return True

MARK_PROVEN(p):
    if p is not already known:
        add p to the proved set
        for each rule waiting for p:
            decrement its remaining count
            if the count is zero: MARK_PROVEN(rule conclusion)

QUERY(q):
    initialize fact/rule indexes and shared tables if absent
    PROVE(q, 0)
    while deferred goals remain: PROVE(next deferred goal, 0)
    return q is a fact or proved
```

An ungrounded cycle cannot create a fact. Pending dependencies allow a cycle
with an external proof to succeed, instead of permanently caching a provisional
failure. Both of these cases passed the separate review tests, as did all 3,645
correct/incorrect candidate queries across the five provided Sudoku puzzles.

**Known issues to resolve before final submission.** In this received version,
repeated premises are counted more than once but waiting rules are stored in a
set, so `(A & A) ==> Q` can incorrectly remain unproved after A is derived.
Premises should be deduplicated before counting. Tables are also retained after
`tell`/`retract`, so the indexes and proof cache must be invalidated when the KB
changes. Finally, expansion is depth-limited but `MARK_PROVEN` still recursively
propagates answers: a 1,500-rule chain raised `RecursionError` in review. Iterative
answer propagation is needed before claiming stack-safe termination for arbitrary
finite Horn KBs. These issues did not invalidate the five supplied puzzle results.
This answer will be updated to describe the corrected implementation after repair.


### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


I use **Naked Pairs**. Consider distinct cells $a,b$ in the same unit (row, column,
or box), and values $p\ne q$. Suppose every other value has been explicitly excluded
from both cells. Define the positive conjunction

$$D_{a,b,p,q}=\bigwedge_{v\notin\{p,q\}}(N_{av}\land N_{bv}).$$

For each other cell $c$ in that unit, add two **separate** definite rules:

$$D_{a,b,p,q}\Rightarrow N_{cp},\qquad D_{a,b,p,q}\Rightarrow N_{cq}.$$

For example, let $a=(1,1)$, $b=(1,2)$, and $\{p,q\}=\{2,7\}$. The body is the
conjunction of the 14 positive Not atoms excluding values $1,3,4,5,6,8,9$ from
each of those two cells. It implies `Not1_3_2` in one rule and `Not1_3_7` in another,
and similarly for columns 4–9 in row 1. Each rule has exactly one positive head.
The body is written out using the existing atoms; $D$ above is only notation,
not a required new proposition.

Why this is sound: in a valid Sudoku the two cells each need a value, both values
come from $\{p,q\}$, and the cells cannot have equal values because they share a
unit. They must therefore use $p$ and $q$ in some order. Neither value can appear
in another cell of that unit. It is enough that their candidate sets are subsets
of this pair; we do not infer unproven candidates by treating missing Not facts as
false, and we do not guess which cell takes which value.

These rules can trigger extra eliminations when last-candidate reasoning alone
stalls. The cost is a much larger KB: a naive 9×9 grounding considers 27 units,
$\binom92$ pairs of cells, $\binom92$ pairs of values, 7 other cells, and 2 heads:
$27\times36\times36\times7\times2=489{,}888$ candidate rules before deduplication.
Each body has 14 premises. This increases construction time, memory, and premise
processing. Rule indexing or generating relevant rules only when needed can help.
Even this extension would not make the rule set sufficient for every Sudoku.
The submitted solver intentionally implements the required elimination and
last-candidate rules; this question develops the extension without silently
changing the measured baseline.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**(a) When backward chaining helps.** In a large KB whose query depends on only a
small subset of rules, indexing rules by conclusion allows backward chaining to
focus on that dependency subgraph. Forward chaining may process many unrelated
facts before reaching the requested conclusion. Index construction itself still
requires reading the KB.

**(b) When it has no advantage.** If most rules are relevant, or many goals require
almost the whole closure, goal-directed expansion may visit nearly everything
and incur additional dependency bookkeeping. A forward implementation that
computes and retains the closure once can be efficient for many queries. Neither
direction is universally faster.

**Measured comparison.** The validation cells above measure both full-grid solvers
on each of the same five puzzles, with a fresh KB for each call. The timed interval
includes KB construction and excludes result assertions and display. The printed
values are actual measurements from this notebook's run, not fixed constants.

The submitted teammate FC function indexes premise lookup on the KB instance,
then calls the teacher's unchanged `pl_fc_entails` for each candidate. That library
function rebuilds its agenda and counters for each query. The BC function retains
proven subgoals and dependency tables across queries on the same KB. Thus a timing
gap includes indexing and reuse differences, not just inference direction.
A single measurement per puzzle is indicative and depends on machine load; it is
not evidence of a universal runtime ordering. These measurements must be rerun
if the solver is changed before submission.


## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

[Current Group 7 Streamlit app](https://daphnezhang627-cell-it5005-group7-sudoku-sudoku-app-lkd5pg.streamlit.app/)

This existing deployment currently runs the earlier verified project, not the
teammate version reviewed in this notebook. Deployment must be updated and checked
after the teammate-code fixes are completed. This notebook remains a review draft.
